# Agent高级用法-流式输出即stream，通过agent.stream(mstream_model=)设置
## 1. 流式输出7中模式（values，updates（默认）、messages、custom、checkpoints、tasks、debug）

### 1.1 values模式:每个步骤执行后都会输出完整状态信息，适合每一步都获取完整状态、状态持久化场景

In [ ]:
from dataclasses import dataclass

from pydantic import Field, BaseModel
from typing import Literal, TypedDict, Dict, Any
from langchain_core.tools import tool
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from rich import print as rprint

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

# 定义工具
@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """

    查询客户基本信息
    Args:
    customer_id: 客户ID，用于唯一标识客户

    Returns:
    包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
    customer_id: 客户ID，用于唯一标识客户

    Returns:
    包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
    包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
    "promotions": ["老用户优惠", "会员专属折扣"],
    "valid_until": "2027-01-31"
    }



model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
    extra_body={"thinking":{"type":"disabled"}}

)
# 创建agent
agent = create_agent(
    model=model,
    tools=[get_current_promotions,check_order_history,query_customer_data],
)


for check in agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="values"
):
    rprint(check)


### 1.2 updates模式:每个步骤执行后，只增量更新状态中发生变化的内容，用于监控Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤。

In [ ]:
for check in agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="updates"
):
    rprint(check)

### 1.3 messages模式:会输出流式返回的Token以及相关的元数据（如：来自哪个节点），可以用在实现类似ChatGPT 的打字机效果场景，为聊天机器人等交互式应用提供最佳的实时体验。

In [ ]:
for check in agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="messages"
):
    rprint(check)

### 1.4 tasks模式:会输出当前task任务开始和结束的时间，包含任务的结果和错误信息，该模式用于监控任务的生命周期。

In [ ]:
for check in agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="tasks"
):
    rprint(check)

### 1.5 debug模式:与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result），该模式用于调试、监控task任务的生命周期。

In [ ]:
for check in agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="debug"
):
    rprint(check)

### 1.6 checkpoints模式:每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景。

In [6]:
from langgraph.checkpoint.memory import InMemorySaver
# 其他工具代码同上，保持不变
# ... ...
# 1. 创建内存检查点存储
checkpointer = InMemorySaver()
# 2. 创建Agent
customer_service_agent = create_agent(
model=model,
tools=[query_customer_data, check_order_history,
get_current_promotions],
checkpointer=checkpointer # 启用检查点
)
# 3. 创建唯一的会话ID
config = {"configurable": {"thread_id": "session01"}}
# 4. 调用Agent
checkpoint_count = 0
# 使用checkpoints模式进行流式监控
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
    config=config,
    stream_mode="checkpoints"
):
    checkpoint_count += 1
    print(f"检查点 #{checkpoint_count}")
    print(chunk)
    print("-" * 50)

检查点 #1
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f197968-b2b7-6b66-bfff-3a5cc3e76c23'}}, 'parent_config': None, 'values': {'messages': []}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': '55fdcec2-3501-a594-3e6c-c67f01e6a505', 'name': '__start__', 'interrupts': (), 'state': None}]}
--------------------------------------------------
检查点 #2
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f197968-b2ba-68ca-8000-babfc0936e98'}}, 'parent_config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f197968-b2b7-6b66-bfff-3a5cc3e76c23'}}, 'values': {'messages': [HumanMessage(content='查询客户ID为 CUST123456 的完整信息和可用优惠', additional_kwargs={}, response_metadata={}, id='b13e98d2-20c9-4f9d-84a5-de378489dad6')]}, 'metadata': {'source': 'loop', 'step': 0, 'parents': {}}, 'next': ['model'], 'tasks': [{'id': 'a1f57cd0-a8ac-

### 1.7 custom模式:开发者通过 get_stream_writer 在工具或节点内部 自定义发送的数据 ，用于 输出 业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标。

In [7]:
import time
from langgraph.config import get_stream_writer


@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()
    writer({"type": "生成销售报告", "message": "开始生成销售报告"})
    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告","message": f"生成销售报告进度百分比：{i *
        25}%"})
    writer({"type": "生成销售报告", "message": "报告生成完成"})
    return f"销售报告：总收入150万元，同比增长12%"

@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告...")
    return "当前库存量为10000件，库存充足，无异常"


# 创建报告生成agent
reporting_agent = create_agent(
model=model,
tools=[generate_sales_report, generate_inventory_report]
)
for chunk in reporting_agent.stream(
{"messages": [{"role": "user","content": "生成销售报告和库存报告"}]},
stream_mode="custom"
):
    print(chunk)
    print("-" * 50)

{'type': '生成销售报告', 'message': '开始生成销售报告'}
--------------------------------------------------
开始库存分析...
--------------------------------------------------
检查当前库存量...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：25%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：50%'}
--------------------------------------------------
生成库存报告...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：75%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '报告生成完成'}
--------------------------------------------------


## 2.0 总结
实现 实时对话交互 ，优先选择messages模式；
观察Agent的 思考与执行步骤 ，优先选择updates模式；
需要查看 每一步状态 优先选择values/tasks/debug模式；
在工具执行时 输出自定义业务 日志优先选择custom模式。